In [30]:
if (!require("tidyverse", quietly = TRUE)) {
  install.packages("tidyverse", quiet = TRUE, verbose = FALSE)
}

if (!require("stringr", quietly = TRUE)) {
  install.packages("tidyverse", quiet = TRUE, verbose = FALSE)
}

if (!require("data.table", quietly = TRUE)) {
  install.packages("data.table", quiet = TRUE, verbose = FALSE)
}

suppressPackageStartupMessages({
    library(tidyverse)
    library(stringr)
    library(data.table)
})

#### Loading Data

In [2]:
# Create a function to read file and select columns
read_and_select <- function(filepath, exclude_col) {
  fread(filepath) %>% select(-all_of(exclude_col))
}

# File paths and columns to exclude
filepaths <- c("../cpnds_genotyping/batch1.tsv", 
               "../cpnds_genotyping/batch2.tsv", 
               "../cpnds_genotyping/batch3.tsv")
exclude_col <- "MENDEL"

# Read files
data_list <- lapply(filepaths, function(x) read_and_select(x, exclude_col))

# Assign to variables
f1 <- data_list[[1]]
f2 <- data_list[[2]]
f3 <- data_list[[3]]

# Read phenotype and pruned data
phenotype <- read_and_select("../cpnds_genotyping/gr1_removed_high_affection.tsv", "LIAB")
pruned <- fread("../cpnds_genotyping/result.pruneddata_0.8.prune.in", header=FALSE) %>% setNames("rsid")
six_genes <- fread("../rsidSixGenesCPNDS.txt", header = FALSE) %>% setNames("rsid")
jagsetAnnot <- fread("../jag.set.annot.annot")
vep <- fread("../feature_tsv/VEP_noncanonical_included_Output.tsv") %>% select(id, cadd_phred, variant_allele) %>% rename(allele = variant_allele, marker = id) %>% na.omit()

#### Examinning Raw Post-QCed Genotype table

In [3]:
genotype <- rbind(f1,f2,f3) %>% 
distinct()

In [4]:
class(str_subset(genotype$ALLELE1, "^$|NA| "))

[1] "character"

In [5]:
#Check missing data in Allele 
sum(length(str_subset(genotype$ALLELE1, "^$|NA| ")))
sum(length(str_subset(genotype$ALLELE2, "^$|NA| ")))

[1] 207

[1] 207

In [6]:
# Store rows with missing data
dropped_rows <- genotype %>%
mutate(ALLELE1 = str_replace(ALLELE1, "^$|NA| ", NA_character_), 
       ALLELE2 = str_replace(ALLELE2, "^$|NA| ", NA_character_)) %>%
filter(is.na(ALLELE1) | is.na(ALLELE2))

# Remove Markers without Alleles 1, 2, or both
genotype <- genotype %>%
mutate(ALLELE1 = str_replace(ALLELE1, "^$|NA| ", NA_character_), 
       ALLELE2 = str_replace(ALLELE2, "^$|NA| ", NA_character_),) %>% 
drop_na(ALLELE1, ALLELE2)

In [7]:
print(paste("the percentage of subjects missing allele information is", round(n_distinct(dropped_rows$SUBJECT)*100/n_distinct(genotype$SUBJECT),2), "%")) 
print(paste("the percentage of markers missing allele information is " ,  round(n_distinct(dropped_rows$MARKER)*100/n_distinct(genotype$MARKER),2), "%")) 

[1] "the percentage of subjects missing allele information is 36.33 %"
[1] "the percentage of markers missing allele information is  28.48 %"


#### identify reference allele: majority of ALLELE2 for each subject

In [8]:
reference <- genotype %>% 
group_by(MARKER) %>%
count(ALLELE2) %>% 
arrange(MARKER, desc(n)) %>%
slice_head(n = 1) %>% 
ungroup %>% 
select(-n) %>% 
rename(reference=ALLELE2)

In [9]:
genotype <- left_join(genotype, reference, by="MARKER")
pruned_genotype <- genotype %>% 
  filter(MARKER %in% pruned$rsid )

#### Convert affection status (phenotypes) to binary

In [10]:
str_subset(phenotype$phenotype, "^$|NA| ")

character(0)

In [11]:
phenotype <- phenotype %>% 
  mutate(phenotype = 
           case_when(
             grepl('healthy', AFFSTAT) ~ 0, 
             grepl('affected', AFFSTAT) ~ 1 
           )) %>% 
  select(-AFFSTAT) %>% 
  rename("subject" = "SUBJECT")

#### Joining VEP Table

In [12]:
# Function to clean marker names
clean_marker <- function(marker) {
  marker <- tolower(marker)
  marker <- gsub("gsa-|seq-", "", marker)
  marker <- dplyr::recode(marker, "ilmnseq_6:35378798" = "rs9658134")
  return(marker)
}

# Convert genotype to long format and clean marker names
genotype_long <- genotype %>% 
  pivot_longer(cols = -c(SUBJECT, MARKER), names_to = "holder", values_to = "allele") %>% 
  select(-holder) %>% 
  rename(marker = MARKER) %>% 
  mutate(marker = clean_marker(marker))

# Combine VEP with genotype in long format
combined <- left_join(genotype_long, vep, by = c("marker", "allele")) %>% 
  mutate(cadd_phred = replace_na(cadd_phred, 0)) %>% 
  rename(subject = SUBJECT)

### Filter for only variants in the 6 genes _PRKCD,PIK3R2, AGT, LRP5, CSNK1A1, PPARD_ 
* (EntrezID: 5580, 5296, 183, 4041, 1452, 5467)

In [13]:
jagsetAnnot <- jagsetAnnot %>% 
mutate(V1 = clean_marker(V1)) %>% 
filter(V3 %in% c("WNT", "IL6")) %>% 
       filter(V2 %in% c("5580", "5296", "183", "4041", "1452", "5467") ) %>% 
       mutate(symbols = case_when(
           grepl("5580", V2) ~ "PRKCD",
           grepl("5296", V2) ~ "PIK3R2",
           grepl("183", V2) ~ "AGT",
           grepl("4041", V2) ~ "LRP5",
           grepl("1452", V2) ~ "CSNK1A1",
           grepl("5467", V2) ~ "PPARD"
  )) %>% 
       setNames(c("marker", "entrezId", "pathways", "gene_symbols"))

# merge dataframes by rsid
combined <- left_join(combined, jagsetAnnot, by = "marker")

In [14]:
head(combined,1)

subject,marker,allele,cadd_phred,entrezId,pathways,gene_symbols
<chr>,<chr>,<chr>,<dbl>,<int>,<chr>,<chr>
LON400276,rs4713859,T,0,5467,WNT,PPARD


* Remove the rsIDs that were not used in the Gene Set Enrichment Analysis

In [15]:
# merge dataframes by rsid
remove <- left_join(combined %>% filter(marker  %in%  c('rs7761870','rs6457821','rs112012380','rs3823433','rs6457813','rs9658134')), jagsetAnnot, by = "marker")
remove  <- remove %>% mutate(variants = paste0(gene_symbols.x, "_", marker)) %>% select(variants)

In [16]:
variant_remove <- remove %>% 
distinct(variants)

remove_list <- as.list(variant_remove$variants)

remove_list

[[1]]
[1] "PPARD_rs7761870"

[[2]]
[1] "PPARD_rs6457821"

[[3]]
[1] "AGT_rs112012380"

[[4]]
[1] "PPARD_rs3823433"

[[5]]
[1] "PPARD_rs6457813"

[[6]]
[1] "PPARD_rs9658134"

#### Reformat Cadd_phred to capture max score for each gene (i.e. the more deleterious allele) for each allele in each subject

In [17]:
# Calculate the maximum cadd_phred score for each subject and gene_symbols pair
df_max <- combined %>% 
  group_by(subject, gene_symbols) %>% 
  summarise(max_CADD = max(cadd_phred), .groups = "drop") %>%  # .groups = "drop" ungroups the data
  pivot_wider(names_from = gene_symbols, values_from = max_CADD, names_prefix = "CADD_max_")

# Merge the maximum cadd_phred score back to the original dataframe
combined <- left_join(combined, df_max, by = "subject")

#### Combine Clinical Covariants

In [18]:
# Load high covariates file and rename 'IID' to 'subject'
cov <- fread("../cpnds_genotyping/high_covariates.tsv") %>% 
  select(-FID) %>% 
  rename(subject = IID)

In [19]:
# Identify individual without Radiation_status_N
cov%>% 
  filter(is.na(RADIATION_STATUS_N)) %>% 
  select(subject)
#Remove the only individual that does not have the radiation_status
cov <- cov %>% 
  filter(subject != 'VAN100373') %>%
  distinct()

combined <- left_join(combined %>% filter(subject != 'VAN100373'), cov, by="subject") %>%
  select(-c(cadd_phred, gene_symbols,allele, entrezId, pathways, marker))

subject
<chr>
VAN100373


#### Format genotype table to wider format

In [20]:
pruned_genotype_wide <- pruned_genotype_wide[,!names(pruned_genotype_wide) %in% remove_list] 

ERROR: Error in eval(expr, envir, enclos): object 'pruned_genotype_wide' not found


In [21]:
pruned_genotype_wide <- pruned_genotype %>% 
  mutate(
    MARKER = clean_marker(MARKER), 
    subject = SUBJECT,
    score = case_when(
      ALLELE1 == ALLELE2 & ALLELE2 == reference ~ 0,
      ALLELE1 != ALLELE2 & ALLELE2 == reference ~ 1,
      ALLELE1 != reference & ALLELE2 != reference ~ 2
    )
  ) %>% 
  left_join(jagsetAnnot %>% select('marker', 'gene_symbols'), by = c("MARKER" = "marker")) %>%
  mutate(variants = paste0(gene_symbols, "_", MARKER)) %>%
  select(subject, variants, score) %>%
  pivot_wider(names_from = variants, values_from = score)

__For some subjects without variants,  the subject is removed from the dataset__

In [22]:
pruned_genotype_wide <- pruned_genotype_wide %>% 
na.omit()

In [34]:
sum(apply(pruned_genotype_wide, 2, function(x) x == ""))

[1] 0

In [35]:
summary(is.na(pruned_genotype_wide))

  subject        PPARD_rs4713859 LRP5_rs3736228  CSNK1A1_rs10476909
 Mode :logical   Mode :logical   Mode :logical   Mode :logical     
 FALSE:192       FALSE:192       FALSE:192       FALSE:192         
 CSNK1A1_rs1379544 CSNK1A1_rs447950 CSNK1A1_rs414582 CSNK1A1_rs3733658
 Mode :logical     Mode :logical    Mode :logical    Mode :logical    
 FALSE:192         FALSE:192        FALSE:192        FALSE:192        
 CSNK1A1_rs2400891 CSNK1A1_rs10062536 LRP5_rs4988300  PPARD_rs4713858
 Mode :logical     Mode :logical      Mode :logical   Mode :logical  
 FALSE:192         FALSE:192          FALSE:192       FALSE:192      
 CSNK1A1_rs1363629 CSNK1A1_rs6861564 PPARD_rs4713854 CSNK1A1_rs7728256
 Mode :logical     Mode :logical     Mode :logical   Mode :logical    
 FALSE:192         FALSE:192         FALSE:192       FALSE:192        
 CSNK1A1_rs452366 LRP5_rs474742   CSNK1A1_rs476741 CSNK1A1_rs241280
 Mode :logical    Mode :logical   Mode :logical    Mode :logical   
 FALSE:192        FALSE:

#### Join previous combined table with pruned genotype table

In [24]:
class(pruned_genotype_wide$LRP5_rs3736228)

[1] "numeric"

In [25]:
pruned_combined <- pruned_genotype_wide  %>% 
left_join(combined, by="subject") %>% 
left_join(phenotype, by="subject")

In [26]:
sum(is.na(pruned_combined))

[1] 0

#### Remove redundancy

In [27]:
pruned_combined <- pruned_combined %>% 
distinct() %>% 
select(-subject)

In [36]:
pruned_combined

PPARD_rs4713859,LRP5_rs3736228,CSNK1A1_rs10476909,CSNK1A1_rs1379544,CSNK1A1_rs447950,CSNK1A1_rs414582,CSNK1A1_rs3733658,CSNK1A1_rs2400891,CSNK1A1_rs10062536,LRP5_rs4988300,⋯,CADD_max_PPARD,CADD_max_PRKCD,SEX_N_MALE,RADIATION_STATUS_N,AGE_MTX_INITIATION_YEAR,PC1,PC2,PC3,PC4,phenotype
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
0,0,1,1,0,0,1,1,0,0,⋯,13.94,6.272,0,0,10.9835616,-0.00819109,-0.01867980,0.005814860,-0.007582830,0
0,1,1,0,1,0,0,0,0,1,⋯,13.94,6.272,0,0,7.9972678,-0.00818886,-0.01854910,0.008773370,-0.008294950,0
0,0,0,1,1,1,1,1,1,0,⋯,13.94,6.272,1,1,16.4986301,-0.00045870,-0.01312030,0.008233090,0.012657200,0
0,0,1,1,0,0,2,2,0,1,⋯,13.94,6.272,0,0,1.8027397,-0.00804984,-0.01916960,0.007358760,-0.009525170,0
1,1,1,2,0,0,2,2,0,2,⋯,13.94,6.272,0,0,10.8657534,-0.00913200,-0.01244720,-0.005811660,-0.005802180,0
0,0,1,1,1,0,0,1,0,1,⋯,13.94,6.272,1,1,1.8794521,-0.00782076,-0.01916950,0.008494100,-0.007924700,0
0,0,0,1,0,0,1,1,0,1,⋯,13.94,6.272,1,0,1.2240437,-0.00858358,-0.01866460,0.007065230,-0.006844470,0
0,0,0,1,0,0,1,1,0,1,⋯,13.94,6.272,0,0,2.4767123,-0.00556460,-0.01578990,0.008940870,0.012408200,0
0,0,0,1,0,1,1,1,0,0,⋯,13.94,12.370,0,0,4.9369863,-0.00838257,-0.01868390,0.008829320,-0.006979120,0


#### Save pruned_features_table

In [37]:
write.table(pruned_combined , "../feature_tsv/feature_pruned_subject_removed.tsv", sep = '\t', row.names = FALSE)